# 🧠 D-SIT: Dopaminergic Spiking Image Transformer
**Dopamine-Modulated Proximal Surrogate Gradient Descent on CIFAR-100**

---

> **Author:** Siddhesh Uttarwar — Brain-inspired Computing Lab, UCSB

This notebook trains the D-SIT architecture on CIFAR-100 using a Google Colab T4 GPU.

### Before You Start
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run all cells sequentially

## Step 1: Verify GPU

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

## Step 2: Clone the Repository

In [ ]:
!git clone https://github.com/SiddheshUttarwar/PSGD-D-SpikingImageTransformer.git
%cd PSGD-D-SpikingImageTransformer

## Step 3: Install Dependencies

In [ ]:
!pip install datasets tqdm -q

## Step 4: Smoke Test — Verify Model Builds

In [ ]:
import sys
sys.path.insert(0, 'd-sit')

from model import DSIT
from optimizer import ProximalAdamW
import torch.nn as nn

# Build the model with CIFAR-100 config
model = DSIT(
    num_classes=100,
    embed_dim=256,
    depth=8,
    num_heads=8,
    T=4,
    img_size=32
)

# Quick forward pass test
x = torch.randn(2, 3, 32, 32)
logits = model(x)
print(f"✅ Forward pass OK — Output shape: {logits.shape}")

# Quick backward pass test
labels = torch.tensor([5, 10])
loss = nn.CrossEntropyLoss()(logits, labels)
loss.backward()
print(f"✅ Backward pass OK — Loss: {loss.item():.4f}")

# Verify dopamine tracker
model.d_tracker.update(labels, logits.detach())
print(f"✅ Dopamine D(t): {model.d_tracker.get_D():.6f} (near 0 = exploration mode)")

param_count = sum(p.numel() for p in model.parameters()) / 1e6
print(f"✅ Model size: {param_count:.2f}M parameters")

del model, x, logits  # Free memory
torch.cuda.empty_cache()

## Step 5: Train on CIFAR-100

This will:
- Download CIFAR-100 from HuggingFace (~170MB)
- Build a D-SIT with `embed_dim=256, depth=8, heads=8` (~7.7M params)
- Train with mixed precision + gradient accumulation (effective batch=64)
- Save `dsit_best.pth` and `dsit_latest.pth` for checkpointing
- Log head pruning status every 5 epochs

In [ ]:
!python d-sit/train.py \
    --dataset cifar100 \
    --img_size 32 \
    --batch_size 16 \
    --accum_steps 4 \
    --embed_dim 256 \
    --depth 8 \
    --num_heads 8 \
    --T 4 \
    --lr 1e-3 \
    --prox_lambda 1e-4 \
    --epochs 100

## Step 6 (If Disconnected): Resume Training

If your Colab session disconnects, re-run Steps 1-3, then run this cell:

In [ ]:
# Only run this if you got disconnected and need to resume
!python d-sit/train.py \
    --dataset cifar100 \
    --img_size 32 \
    --batch_size 16 \
    --accum_steps 4 \
    --embed_dim 256 \
    --depth 8 \
    --num_heads 8 \
    --T 4 \
    --lr 1e-3 \
    --prox_lambda 1e-4 \
    --epochs 100 \
    --resume dsit_latest.pth

## Step 7: Analyze Head Pruning

After training, inspect which attention heads were pruned by the proximal optimizer:

In [ ]:
import sys
sys.path.insert(0, 'd-sit')
import torch
from model import DSIT
from optimizer import ProximalAdamW

# Load best checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DSIT(num_classes=100, embed_dim=256, depth=8, num_heads=8, T=4, img_size=32).to(device)

ckpt = torch.load('dsit_best.pth', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded best checkpoint from epoch {ckpt['epoch']+1}")
print(f"Best Val Acc: {ckpt['best_val_acc']*100:.2f}%")
print(f"Final D(t): {ckpt['D_t']:.6f}")

# Analyze head norms
print("\n" + "="*50)
print("ATTENTION HEAD PRUNING ANALYSIS")
print("="*50)

total_alive = 0
total_heads = 0
for i, blk in enumerate(model.blocks):
    w = blk.attn.out_proj.weight
    w_reshaped = w.view(8, -1)
    norms = torch.norm(w_reshaped, p=2, dim=1)
    alive = (norms > 1e-6).sum().item()
    total_alive += alive
    total_heads += 8
    status = ' '.join(['🟢' if n > 1e-6 else '🔴' for n in norms])
    print(f"Block {i:2d}: {status}  ({alive}/8 alive, max_norm={norms.max():.4f})")

print(f"\nTotal: {total_alive}/{total_heads} heads alive ({100*total_alive/total_heads:.1f}%)")

## Step 8: Save Checkpoint to Google Drive (Optional)

Persist your trained model across Colab sessions:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

save_dir = '/content/drive/MyDrive/DSIT_checkpoints'
os.makedirs(save_dir, exist_ok=True)

for f in ['dsit_best.pth', 'dsit_latest.pth']:
    if os.path.exists(f):
        shutil.copy(f, os.path.join(save_dir, f))
        print(f"✅ Saved {f} to Google Drive")
    else:
        print(f"⚠️ {f} not found")

print(f"\nCheckpoints saved to: {save_dir}")